# 🎬 Quarantine Gate 0: 2036 — AI Live-Action Video Studio (Google AI Studio Veo 3 / Colab GPU)

本 Notebook 自动拉取 [`evanzhousy/interactive-video-game`](https://github.com/evanzhousy/interactive-video-game) 仓库中的 **28 张 35mm 真人防爆窗关键帧** 与 **40 条零口型（Zero-Lip-Sync）I2V 提示词 (`public/media-manifest.json`)**，支持两种生成模式：

- **模式 A (`PROVIDER = 'veo'`) — Google AI Studio (Veo 3)**：无需重算力，点击 Colab 左侧 `🔑 Secrets` → **Import key from Google AI Studio**（命名为 `GEMINI_API_KEY`）即可调用 `veo-3.0-generate-001` 首尾帧锚定生成电影级实拍视频。
- **模式 B (`PROVIDER = 'ltx_gpu'`) — Colab 免费 T4 / L4 GPU 开源实拍视频模型**：零 API 费用，将运行时切换为 `Runtime -> Change runtime type -> T4 GPU`，使用开源 `Lightricks/LTX-Video` 图生视频管线直接渲染。

生成完成后，运行最后一格即可自动保存到你的 Google Drive `MyDrive/quarantine-gate-0-videos/`，随后在对话中告诉 Agent 一声，Agent 会直接从网盘拉回全部 `.mp4` 并自动发布到线上 GitHub Pages！

In [ ]:
#@title 1. 克隆仓库并加载 40 条 35mm 真人关键帧清单
!rm -rf /content/interactive-video-game
!git clone --depth 1 https://github.com/evanzhousy/interactive-video-game.git /content/interactive-video-game
%cd /content/interactive-video-game

import json, pathlib
manifest = json.loads(pathlib.Path('public/media-manifest.json').read_text())
print(f"Loaded {manifest['counts']['visitors']} survivors · {manifest['counts']['videoClips']} I2V video specs")

In [ ]:
#@title 2. 选择渲染引擎与筛选范围
PROVIDER = 'veo' #@param ['veo', 'ltx_gpu']
VISITOR_FILTER = 'ALL' #@param ['ALL', 'V-01', 'V-02', 'V-03', 'V-04', 'V-05', 'V-06', 'V-07', 'V-08']
STATE_FILTER = 'ALL' #@param ['ALL', 'idle', 'flinch', 'lean_in', 'admit', 'purge']
USE_END_FRAME_ON_VEO = True #@param {type:'boolean'}

selected_clips = [
    c for c in manifest['videoClips']
    if (VISITOR_FILTER == 'ALL' or c['visitorId'] == VISITOR_FILTER)
    and (STATE_FILTER == 'ALL' or c['state'] == STATE_FILTER)
]
print(f'Selected {len(selected_clips)} clips via provider={PROVIDER}:')
for c in selected_clips[:8]:
    print(' •', c['visitorId'], c['state'], '->', c['targetMp4'])

In [ ]:
#@title 3A. 若选择 Google AI Studio (`veo`)：读取 `GEMINI_API_KEY` 并调用 Veo 3 渲染
import os, time, subprocess, pathlib

def normalize_mp4(raw_path: str, out_path: str):
    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', raw_path,
        '-vf', 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2',
        '-r', '30', '-c:v', 'libx264', '-profile:v', 'high', '-pix_fmt', 'yuv420p',
        '-crf', '21', '-movflags', '+faststart', '-an', out_path
    ], check=True)
    os.remove(raw_path)

if PROVIDER == 'veo':
    !pip install -q -U google-genai pillow
    from google import genai
    from google.genai import types
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except Exception:
        api_key = os.environ.get('GEMINI_API_KEY')
    if not api_key:
        raise RuntimeError('请先在 Colab 左侧 🔑 Secrets 中添加 GEMINI_API_KEY（可直接点击 Import key from Google AI Studio）！')

    client = genai.Client(api_key=api_key)
    model_name = 'veo-3.0-generate-001'

    for idx, spec in enumerate(selected_clips, 1):
        out_mp4 = spec['targetMp4']
        if os.path.exists(out_mp4) and os.path.getsize(out_mp4) > 4096:
            print(f'↷ [{idx}/{len(selected_clips)}] Skip existing {out_mp4}')
            continue
        print(f'⏳ [{idx}/{len(selected_clips)}] Veo generating {spec["visitorId"]} ({spec["state"]})...')
        first_bytes = pathlib.Path(spec['firstFramePortrait']).read_bytes()
        cfg_kwargs = {'aspect_ratio': '16:9', 'negative_prompt': spec['negativePrompt']}
        if USE_END_FRAME_ON_VEO and spec['state'] != 'idle':
            end_bytes = pathlib.Path(spec['referenceEndFrame']).read_bytes()
            cfg_kwargs['last_frame'] = types.Image(image_bytes=end_bytes, mime_type='image/jpeg')

        op = client.models.generate_videos(
            model=model_name,
            prompt=spec['prompt'],
            image=types.Image(image_bytes=first_bytes, mime_type='image/jpeg'),
            config=types.GenerateVideosConfig(**cfg_kwargs),
        )
        while not op.done:
            time.sleep(8)
            op = client.operations.get(op)

        vid = op.response.generated_videos[0].video
        client.files.download(file=vid)
        tmp_raw = out_mp4 + '.raw.mp4'
        pathlib.Path(tmp_raw).parent.mkdir(parents=True, exist_ok=True)
        vid.save(tmp_raw)
        normalize_mp4(tmp_raw, out_mp4)
        print(f'✓ Saved {out_mp4} ({os.path.getsize(out_mp4)//1024} KB)')
else:
    print('当前选择为 ltx_gpu，请运行下一格 3B。')

In [ ]:
#@title 3B. 若选择 Colab 免费 T4 GPU (`ltx_gpu`)：使用开源 Lightricks/LTX-Video 图生视频渲染
if PROVIDER == 'ltx_gpu':
    !pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg
    import os, pathlib, torch
    from PIL import Image
    from diffusers import LTXImageToVideoPipeline
    from diffusers.utils import export_to_video

    if not torch.cuda.is_available():
        raise RuntimeError('未检测到 GPU！请在顶部菜单点击 Runtime -> Change runtime type -> 选择 T4 GPU。')

    pipe = LTXImageToVideoPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=torch.bfloat16
    )
    pipe.enable_model_cpu_offload()

    for idx, spec in enumerate(selected_clips, 1):
        out_mp4 = spec['targetMp4']
        if os.path.exists(out_mp4) and os.path.getsize(out_mp4) > 4096:
            print(f'↷ [{idx}/{len(selected_clips)}] Skip existing {out_mp4}')
            continue
        print(f'⏳ [{idx}/{len(selected_clips)}] LTX-Video GPU rendering {spec["visitorId"]} ({spec["state"]})...')
        init_img = Image.open(spec['firstFramePortrait']).convert('RGB').resize((768, 448))
        num_frames = 97 if spec['state'] == 'idle' else 73
        frames = pipe(
            image=init_img,
            prompt=spec['prompt'],
            negative_prompt=spec['negativePrompt'],
            width=768,
            height=448,
            num_frames=num_frames,
            num_inference_steps=32,
        ).frames[0]
        tmp_raw = out_mp4 + '.raw.mp4'
        pathlib.Path(tmp_raw).parent.mkdir(parents=True, exist_ok=True)
        export_to_video(frames, tmp_raw, fps=24)
        normalize_mp4(tmp_raw, out_mp4)
        print(f'✓ Saved {out_mp4} ({os.path.getsize(out_mp4)//1024} KB)')
else:
    print('当前选择为 veo，已由上一格 3A 处理。')

In [ ]:
#@title 4. 导出生成好的 `.mp4` 到 Google Drive（供 Agent 一键拉回部署）或下载 ZIP
SAVE_TO_GOOGLE_DRIVE = True #@param {type:'boolean'}
DOWNLOAD_ZIP = False #@param {type:'boolean'}

import shutil, pathlib
mp4_files = list(pathlib.Path('public/videos').rglob('*.mp4'))
print(f'Found {len(mp4_files)} rendered MP4 clips.')

if SAVE_TO_GOOGLE_DRIVE and mp4_files:
    from google.colab import drive
    drive.mount('/content/drive')
    dst_root = pathlib.Path('/content/drive/MyDrive/quarantine-gate-0-videos')
    for f in mp4_files:
        rel = f.relative_to('public/videos')
        target = dst_root / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)
    print(f'✓ Synced {len(mp4_files)} clips to Google Drive: MyDrive/quarantine-gate-0-videos/')
    print('现在只需回到聊天窗口告诉 Agent：「视频已存到 Drive 的 quarantine-gate-0-videos」，Agent 将自动拉回并上线！')

if DOWNLOAD_ZIP and mp4_files:
    shutil.make_archive('/content/gate0-videos', 'zip', 'public/videos')
    from google.colab import files
    files.download('/content/gate0-videos.zip')